### 다중 쿼리 생성 리트리버 ( MultiQueryRetriever )

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:

from langchain_teddynote import logging

logging.langsmith("test0914")

LangSmith 추적을 시작합니다.
[프로젝트명]
test0914


In [4]:
# 샘플 벡터DB 구축
from langchain_community.document_loaders import WebBaseLoader
from langchain_classic.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
import bs4

# 블로그 포스트 로드
loader = WebBaseLoader(
    "https://teddylee777.github.io/openai/openai-assistant-tutorial/", encoding="utf-8"
)

# 문서 분할
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 0)
docs = loader.load_and_split(text_splitter)

# 임베딩 정의
openai_embedding = OpenAIEmbeddings()

# 벡터 DB 생성
db = FAISS.from_documents(docs, openai_embedding)

# retriever 생성
retriever = db.as_retriever()

# 문서 검색
query = "OpenAI Assistant API 의 Function 사용법에 대해 알려주세요"
relevant_dcos = retriever.invoke(query)

# 검색된 문서의 개수 출력
len(relevant_dcos)

4

In [5]:
print(relevant_dcos[1].page_content)

가장 강력한 도구로서, Assistant에게 사용자 정의 함수를 지정할 수 있습니다. 이는 Chat Completions API에서의 함수 호출과 매우 유사합니다.


Function calling(함수 호출) 도구를 사용하면 Assistant 에게 사용자 정의 함수 를 설명하여 호출해야 하는 함수를 인자와 함께 지능적으로 반환하도록 할 수 있습니다.


Assistant API는 실행 중에 함수를 호출할 때 실행을 일시 중지하며, 함수 호출 결과를 다시 제공하여 Run 실행을 계속할 수 있습니다. (이는 사용자 피드백을 받아 재게할 수 있는 의미이기도 합니다. 아래 튜토리얼에서 상세히 다룹니다).


- 사용 방법
- MultiQueryRetriever 에 사용할 LLM을 지정하고 질의 생성에 사용하면 
- retriever가 나머지 작업을 처리

In [6]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(temperature=0, model="gpt-4o-mini")

multiquery_retriever = MultiQueryRetriever.from_llm(
    # MultiQueryRetriever를 언어 모델을 사용하여 초기화
    # 벡터 데이터베이스의 retriever와 언어 모델을 전달
    retriever= db.as_retriever(),
    llm= llm
)

- 아래는 다중 쿼리를 생성하는 중간 과정을 디버깅하기 위하여 실행하는 코드

In [7]:
# 쿼리에 대한 로깅 설정
import logging

logging.basicConfig()
logging.getLogger("langchain.retriever.multi_query").setLevel(logging.INFO)

- retriever_from_llm 객체의 invoke 메서드를 사용하여 주어진 question과 관련된 문서를 검색

In [8]:
# 질문을 정의
question = "OpenAI Assistant API의 Functions 사용법에 대해 알려주세요"

# 문서 검색
relevant_dcos = multiquery_retriever.invoke(question)

# 검색된 고유한 문서의 개수를 반환
print(
    f"============\n검색된 문서 개수 : {len(relevant_dcos)}",
    end="\n========================\n"
)

# 검색된 문서의 내용을 출력
print(relevant_dcos[0].page_content)

검색된 문서 개수 : 4
OpenAI의 새로운 Assistants API는 대화와 더불어 강력한 도구 접근성을 제공합니다. 본 튜토리얼은 OpenAI Assistants API를 활용하는 내용을 다룹니다. 특히, Assistant API 가 제공하는 도구인 Code Interpreter, Retrieval, Functions 를 활용하는 방법에 대해 다룹니다. 이와 더불어 파일을 업로드 하는 내용과 사용자의 피드백을 제출하는 내용도 튜토리얼 말미에 포함하고 있습니다.



주요내용


- LCEL Chain 활용하는 방법
- 사용자 정의 프롬프트 정의하고, 정의한 프롬프트와 함께 Chain을 생성
- Chain은 사용자의 질문을 입력 받으면 질문을 생성한뒤, 생성된 질문을 반환

In [9]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 프롬프트 템플릿을 정의
prompt = PromptTemplate.from_template(
    """You are an AI language model assistant. 
Your task is to generate five different versions of the given user question to retrieve relevant documents from a vector database. 
By generating multiple perspectives on the user question, your goal is to help the user overcome some of the limitations of the distance-based similarity search. 
Your response should be a list of values separated by new lines, eg: `foo\nbar\nbaz\n`

#ORIGINAL QUESTION: 
{question}

#Answer in Korean:
"""
)

llm = ChatOpenAI(temperature=0, model="gpt-4o-mini")

# LLMChain을 생성
custom_multiquery_chain = (
    {"question" : RunnablePassthrough()} | prompt | llm | StrOutputParser()
)

# 질문을 정의
question = "OpenAI Assistant API의 Functions 사용법에 대해 알려주세요."

# 체인을 실행하여 생성된 다중 쿼리를 확인
multi_queries = custom_multiquery_chain.invoke(question)

#결과를 확인
multi_queries

'OpenAI Assistant API의 Functions 기능을 사용하는 방법을 설명해 주세요.  \nOpenAI Assistant API에서 Functions를 활용하는 방법에 대해 알고 싶습니다.  \nOpenAI Assistant API의 Functions를 어떻게 사용할 수 있는지 알려주세요.  \nOpenAI Assistant API의 Functions 사용에 대한 가이드를 제공해 주세요.  \nOpenAI Assistant API의 Functions 기능에 대한 자세한 정보를 원합니다.  '

In [10]:
# 이전에 생성한 Chain을 MultiQueryRetriever에 전달하여 retriever할 수 있음

multiquery_retriever = MultiQueryRetriever.from_llm(
    llm= custom_multiquery_chain,
    retriever= db.as_retriever()
)

In [11]:
# 문서를 검색하고 결과를 확인

relevant_dcos = multiquery_retriever.invoke(question)

print(
    f"================\n 검색된 문서 개수 : {len(relevant_dcos)}",
    end="\n=====================\n"
)

print(relevant_dcos[0].page_content)

 검색된 문서 개수 : 4
OpenAI의 새로운 Assistants API는 대화와 더불어 강력한 도구 접근성을 제공합니다. 본 튜토리얼은 OpenAI Assistants API를 활용하는 내용을 다룹니다. 특히, Assistant API 가 제공하는 도구인 Code Interpreter, Retrieval, Functions 를 활용하는 방법에 대해 다룹니다. 이와 더불어 파일을 업로드 하는 내용과 사용자의 피드백을 제출하는 내용도 튜토리얼 말미에 포함하고 있습니다.



주요내용
